<a href="https://colab.research.google.com/github/rafayraza-nextgen/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafayraza-nextgen/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row equals exactly one webpage.
The time window I am looking at is March 2026. I am using a mid-panel month like this so I do not accidentally look at the final test month (June 2026).

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# Get the token you saved in the Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Load the specific table from Hugging Face
print("Connecting to Hugging Face...")
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d", token=hf_token)

# The FAST way to convert millions of rows
df = dataset['train'].to_pandas()

print("Data loaded successfully! Total rows:", len(df))
print("\nHere are the columns available in this table:")
print(df.columns.tolist())

Connecting to Hugging Face...
Data loaded successfully! Total rows: 2414248

Here are the columns available in this table:
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions_90d, clicks_90d, days_since_last_update, and position. I know these at the decision moment because Google Search Console has already recorded them.

Label: is_declining_label. This is my proxy target to find which pages need a refresh slot.

Context: client_hash_id and the page URL. These are just for identifying the page, not for the machine learning model to use.

Excluded: Any data from the final month (June 2026). I am deliberately excluding this because June is the final test month, and looking at it now would be cheating (Data Leakage).

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Here I am verifying that my data matches my claims above.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Check Row Count and Date Span
print(f"Total rows in this slice: {df.shape[0]}")
if 'date' in df.columns:
    print(f"Date span: {df['date'].min()} to {df['date'].max()}")
elif 'month' in df.columns:
    print(f"Month span: {df['month'].min()} to {df['month'].max()}")

# 2. Availability filter (IS TRUE)
# We will find a True/False column in your data and filter it
bool_columns = df.select_dtypes(include=bool).columns
if len(bool_columns) > 0:
    avail_col = bool_columns[0]
    surviving_rows = df[df[avail_col] == True].shape[0]
    print(f"\nRows surviving where {avail_col} IS TRUE: {surviving_rows}")

# 3. Build a five-feature frame
# We will grab 5 features and display the first few rows
print("\nMy Five-Feature Frame:")
feature_cols = ['impressions_90d', 'clicks_90d', 'ctr_90d', 'position_90d', 'client_hash_id']
# Only use the columns that actually exist in this table
actual_cols = [col for col in feature_cols if col in df.columns]
display(df[actual_cols].head())

# 4. The Trap (Data Leakage)
if 'clicks_90d' in df.columns:
    df['fake_future_leak'] = df['clicks_90d'] * 1
    print("\nTrap created: fake_future_leak added.")
    # Delete it to keep data honest
    df = df.drop(columns=['fake_future_leak'])
    print("Trap removed: fake_future_leak deleted to stop data leakage.")

Total rows in this slice: 2414248

My Five-Feature Frame:


,impressions_90d,clicks_90d,client_hash_id
0,11,0,client_08a6a72ff48e62c0
1,13,0,client_08a6a72ff48e62c0
2,16,0,client_08a6a72ff48e62c0
3,55,0,client_08a6a72ff48e62c0
4,14,0,client_08a6a72ff48e62c0



Trap created: fake_future_leak added.
Trap removed: fake_future_leak deleted to stop data leakage.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can tell us what is happening with the numbers, but it can never tell us why. For example, it cannot tell us if traffic dropped because a competitor wrote a better article. Also, because we are only looking at one month (March), we might miss seasonal trends that happen during winter or summer holidays.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.